In [ ]:
import numpy as np
from scipy.ndimage import label

# 1. Masque binaire des pixels en MHW (ex: sévérité > 1) sur toute la période de validation
# X a pour forme (T, C, H, W). On prend la sévérité réelle dé-standardisée :
val_start = prep["pe"]
truth_val_std = X[val_start:, ti].cpu().numpy()
truth_val_severity = truth_val_std * prep["sds"][ti] + prep["mus"][ti]

# Masque des vagues de chaleur dans l'océan uniquement
mhw_mask_3d = (truth_val_severity > 1) & data["ocean"][None, :, :]

# 2. Détection des événements 3D connexes (voisins en espace ET en temps)
labeled_array, num_events = label(mhw_mask_3d) #c'est vraiment super ça, ça permet de suivre les events d'un jour à l'autre car le temps c'est une troisième dim 

print(f"=== DIAGNOSTIC ===")
print(f"Nombre total d'événements MHW distincts détectés : {num_events}")

# 3. Filtrer les micro-bruits (garder seulement les MHW qui durent > 3 jours et font > 5 pixels)
min_pixels = 15  # Volume spatio-temporel minimal
event_sizes = np.bincount(labeled_array.ravel())
valid_events = np.where(event_sizes >= min_pixels)[0]
valid_events = valid_events[valid_events != 0] # Exclure le fond (0)

print(f"Nombre d'événements MHW significatifs : {len(valid_events)}")

In [ ]:
# ==============================================================================
# FILTRAGE AVANCÉ DES MHW (Norme Océanographique)
# ==============================================================================
filtered_events = []

for event_id in range(1, num_events + 1):
    # Masque booléen pour cet événement précis
    event_mask = (labeled_array == event_id)
    
    # 1. Durée (nombre de jours distincts touchés)
    days_impacted = np.where(event_mask.any(axis=(1, 2)))[0]
    duration = len(days_impacted)
    
    # 2. Sévérité maximale atteinte durant l'événement
    max_sev = truth_val_severity[event_mask].max()
    
    # 3. Surface maximale atteinte sur 1 journée (en nombre de pixels)
    max_surface = event_mask.sum(axis=(1, 2)).max()
    
    # CRITÈRES D'INCLUSION :
    # - Au moins 5 jours
    # - Au moins Sévérité 2 (Vague modérée/forte)
    # - Au moins 10 pixels de surface au pic
    if duration >= 5 and max_sev >= 2.0 and max_surface >= 10:
        filtered_events.append({
            'id': event_id,
            'duration': duration,
            'max_severity': max_sev,
            'max_surface': max_surface,
            'start_day': days_impacted[0] + val_start
        })

print(f"=== RÉSULTAT DU FILTRAGE ===")
print(f"Nombre de MHW physiques majeures conservées : {len(filtered_events)}")

In [ ]:
import pandas as pd
import numpy as np

# 1. On extrait la tranche exacte de dates qui correspond à prep["X"]
dates_X = data["dates"][-len(prep["X"]):]

# 2. Création du DataFrame
df_events = pd.DataFrame(filtered_events)

# 3. Conversion du jour de début en date réelle avec dates_X
df_events["date_debut"] = dates_X[df_events["start_day"]]

# Tri par sévérité maximale décroissante
df_events = df_events.sort_values(by="max_severity", ascending=False).reset_index(drop=True)

print("=== TOP 10 DES MHW LES PLUS INTENSES ===")
display(df_events[['id', 'date_debut', 'duration', 'max_severity', 'max_surface']].head(10))


In [ ]:
# ==============================================================================
# 2. CARTE CUMULÉE DES HOTSPOTS
# ==============================================================================
# Création d'un masque binaire contenant uniquement les 32 événements filtrés
filtered_ids = [e['id'] for e in filtered_events]
mask_32_events = np.isin(labeled_array, filtered_ids)

# Nombre de jours en MHW pour chaque pixel sur toute la période
mhw_days_per_pixel = mask_32_events.sum(axis=0)
mhw_days_per_pixel = np.where(data["ocean"], mhw_days_per_pixel, np.nan)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Carte de fréquence
im0 = ax[0].imshow(mhw_days_per_pixel, cmap="YlOrRd", origin="lower")
ax[0].set_title("Nombre cumulé de jours en MHW (32 events)")
plt.colorbar(im0, ax=ax[0], label="Jours cumulés", fraction=0.046)

# Graphique 2 : Distribution Durée vs Sévérité
sc = ax[1].scatter(df_events["duration"], df_events["max_severity"], 
                   c=df_events["max_surface"], cmap="viridis", s=100, alpha=0.8)
ax[1].set_xlabel("Durée (jours)")
ax[1].set_ylabel("Sévérité maximale")
ax[1].set_title("Durée vs Sévérité (Taille des points = Surface)")
ax[1].grid(True, linestyle="--", alpha=0.5)
plt.colorbar(sc, ax=ax[1], label="Surface max (pixels)")

plt.tight_layout()
plt.savefig("visu.png", dpi=150, bbox_inches="tight")
Image("visu.png")

In [ ]:
def inspect_mhw(event_rank=0):
    """Affiche l'évolution spatio-temporelle de l'événement au rang 'event_rank' dans df_events."""
    dates_X = data["dates"][-len(prep["X"]):]
    
    event_info = df_events.iloc[event_rank]
    event_id = event_info['id']
    
    event_mask = (labeled_array == event_id)
    time_indices = np.where(event_mask.any(axis=(1, 2)))[0] + val_start
    
    n_days = len(time_indices)
    cols = min(5, n_days)
    rows = int(np.ceil(n_days / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), squeeze=False)
    fig.suptitle(f"Événement ID #{event_id} | Rang {event_rank} | Durée: {n_days}j | Sev Max: {event_info['max_severity']:.2f}", fontsize=14)
    
    for i, t in enumerate(time_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        
        sev_day = truth_val_severity[t - val_start]
        sev_day = np.where(data["ocean"], sev_day, np.nan)
        
        im = ax.imshow(sev_day, cmap="YlOrRd", vmin=0, vmax=5, origin="lower")
        
        single_event_day = event_mask[t - val_start]
        ax.contour(single_event_day, levels=[0.5], colors='blue', linewidths=1.5)
        
        # Correction ici : dates_X au lieu de prep["dates"]
        date_str = str(dates_X[t])[:10]
        ax.set_title(f"Jour {i+1} ({date_str})", fontsize=10)
        ax.axis('off')
        
    for i in range(n_days, rows * cols):
        r, c = divmod(i, cols)
        axes[r, c].axis('off')
        
    plt.tight_layout()
    plt.savefig("visu2.png", dpi=150, bbox_inches="tight")
    

# Test sur le premier événement
inspect_mhw(event_rank=0)
Image("visu2.png")